## Dataset's Hyperbolicity Testing

The key metric is Gromov's delta-hyperbolicity (δ), which measures how "tree-like" your data is. A value close to 0 means your data has hierarchical structure that hyperbolic embeddings will exploit well.

In [1]:
import numpy as np
from scipy.spatial import distance_matrix
from tqdm import tqdm
from PIL import Image
import torch
import torchvision.transforms.functional as TF
from rfdetr import RFDETRBase

# --- 1. delta hyperbolicity functions (from hyptorch/delta.py) ---

def delta_hyp(dismat):
    """Computes delta hyperbolicity from a distance matrix."""
    p = 0
    row = dismat[p, :][np.newaxis, :]
    col = dismat[:, p][:, np.newaxis]
    XY_p = 0.5 * (row + col - dismat)
    maxmin = np.max(np.minimum(XY_p[:, :, None], XY_p[None, :, :]), axis=1)
    return np.max(maxmin - XY_p)


def batched_delta_hyp(X, n_tries=10, batch_size=1500):
    """Computes relative delta hyperbolicity with random sampling."""
    vals = []
    for i in tqdm(range(n_tries)):
        idx = np.random.choice(len(X), min(batch_size, len(X)), replace=False)
        X_batch = X[idx]
        distmat = distance_matrix(X_batch, X_batch)
        diam = np.max(distmat)
        if diam > 0:
            delta_rel = 2 * delta_hyp(distmat) / diam
            vals.append(delta_rel)
    return np.mean(vals), np.std(vals)


### 1. Extract decoder features from trained model

In [ ]:
model = RFDETRBase(pretrain_weights="output/checkpoint_best_total.pth")
model.model.model.eval()
device = model.model.device

# Load your dataset images
import os, json, cv2
dataset_dir = "Veiculos-Contar-3"  
with open(os.path.join(dataset_dir, "train", "_annotations.coco.json")) as f:
    ann = json.load(f)

print(f"Train images: {len(ann['images'])}")

# Count annotations per class
from collections import Counter
class_counts = Counter(a["category_id"] for a in ann["annotations"])
class_names = {c["id"]: c["name"] for c in ann["categories"]}
for cid, count in sorted(class_counts.items()):
    print(f"  {class_names[cid]:15s}: {count} annotations")

all_features = []
all_labels = []

# Build image_id -> annotations lookup
img_id_to_anns = {}
for a in ann["annotations"]:
    img_id_to_anns.setdefault(a["image_id"], []).append(a)

with torch.no_grad():
    for img_info in tqdm(ann["images"][:2000]):
        img_path = os.path.join(dataset_dir, "train", img_info["file_name"])
        pil_img = Image.open(img_path).convert("RGB")

        # Preprocess the same way RF-DETR does
        img_tensor = TF.to_tensor(pil_img).to(device)
        img_tensor = TF.normalize(
            img_tensor,
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        )
        img_tensor = torch.nn.functional.interpolate(
            img_tensor.unsqueeze(0),
            size=(model.model.resolution, model.model.resolution),
            mode="bilinear",
        )

        # Forward through backbone + transformer decoder (but NOT class_embed)
        from rfdetr.util.misc import nested_tensor_from_tensor_list
        samples = nested_tensor_from_tensor_list(img_tensor)
        features, poss = model.model.model.backbone(samples)
        srcs = [feat.decompose()[0] for feat in features]
        masks = [feat.decompose()[1] for feat in features]

        refpoint = model.model.model.refpoint_embed.weight[:model.model.model.num_queries]
        query_feat = model.model.model.query_feat.weight[:model.model.model.num_queries]

        hs, _, _, _ = model.model.model.transformer(
            srcs, masks, poss, refpoint, query_feat
        )

        # hs shape: [dec_layers, 1, num_queries, hidden_dim]
        # Take last decoder layer, squeeze batch
        decoder_features = hs[-1].squeeze(0)  # [300, 256]

        # Get the class predictions to filter high-confidence detections
        logits = model.model.model.class_embed(hs[-1]).squeeze(0)  # [300, num_classes]
        scores = logits.sigmoid().max(dim=-1)
        conf = scores.values   # [300]
        # Debug: see what confidence scores look like
        if img_info == ann["images"][0]:
            print(f"Max confidence: {conf.max().item():.4f}")
            print(f"Detections > 0.1: {(conf > 0.1).sum().item()}")
            print(f"Detections > 0.3: {(conf > 0.3).sum().item()}")
        classes = scores.indices  # [300]

                # Lower threshold to capture more detections
        keep = conf > 0.1  # lowered from 0.3

        if keep.sum() > 0:
            all_features.append(decoder_features[keep].cpu().numpy())
            all_labels.append(classes[keep].cpu().numpy())

    # After the loop, check what we got
    print(f"Images processed: {len(ann['images'][:2000])}")
    print(f"Feature batches collected: {len(all_features)}")

    if len(all_features) == 0:
        raise RuntimeError(
            "No detections found. Try using model.predict() on a single image first "
            "to verify the model is working correctly."
        )

all_features = np.concatenate(all_features, axis=0)
all_labels = np.concatenate(all_labels, axis=0)

Reinitializing detection head with 8 classes


Loading pretrain weights
Train images: 5528
  Bus            : 115 annotations
  Motorcycle     : 400 annotations
  Pickup         : 11049 annotations
  Sedan          : 10840 annotations
  Suv            : 2152 annotations
  Truck          : 5248 annotations
  Van            : 1551 annotations


  0%|          | 1/2000 [00:00<16:37,  2.00it/s]

Max confidence: 0.7809
Detections > 0.1: 3
Detections > 0.3: 2


100%|██████████| 2000/2000 [11:23<00:00,  2.93it/s]

Images processed: 2000
Feature batches collected: 2000


In [ ]:
# Compute delta hyperbolicity
mean_delta, std_delta = batched_delta_hyp(all_features, n_tries=30, batch_size=1000)
print(f"Relative delta hyperbolicity: {mean_delta:.6f} +/- {std_delta:.6f}")

100%|██████████| 30/30 [01:43<00:00,  3.44s/it]

Relative delta hyperbolicity: 0.389367 +/- 0.016440


### 2. Test on raw image features (using VGG, uncropped img)

In [ ]:
import torchvision
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms

# Use the same approach as hyptorch/delta.py but with your images
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

vgg = torchvision.models.vgg16(weights="DEFAULT")
vgg_feats = vgg.features
vgg_classifier = nn.Sequential(*list(vgg.classifier.children())[:-1])

class Flatten(nn.Module):
    def forward(self, x):
        return x.view(x.shape[0], -1)

vgg_part = nn.Sequential(vgg_feats, Flatten(), vgg_classifier).to(device)
vgg_part.eval()

# Create a dataset from your vehicle images
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Use torchvision ImageFolder or manually load crops of detected vehicles
# For simplicity, load full test images:
from torchvision.datasets import ImageFolder
# Or load images manually:
all_features = []
for img_info in tqdm(ann["images"][:500]):
    img_path = os.path.join(dataset_dir, "train", img_info["file_name"])
    img = Image.open(img_path).convert("RGB")
    img_t = transform(img).unsqueeze(0).to(device)
    with torch.no_grad():
        feat = vgg_part(img_t).cpu().numpy()
    all_features.append(feat)

all_features = np.concatenate(all_features, axis=0)
mean_delta, std_delta = batched_delta_hyp(all_features, n_tries=20, batch_size=min(1500, len(all_features)))
print(f"Relative delta hyperbolicity (VGG features): {mean_delta:.6f} +/- {std_delta:.6f}")

### 3. Test on raw image features (using VGG, cropped img)

In [10]:
from typing import Any


import os
import numpy as np
from PIL import Image
import torch
from tqdm import tqdm

# Ensure your model is in eval mode
vgg_part.eval()

# 1. Map image IDs to their annotations (if not already done)
img_id_to_anns = {}
for a in ann["annotations"]:
    img_id_to_anns.setdefault(a["image_id"], []).append(a)

all_crop_features = []
crop_labels = [] # Optional: keep track of classes if you want to analyze later

# We'll set a max limit to keep the calculation time reasonable 
# (Gromov delta calculation maxes out its batch size at 1500 anyway)
MAX_CROPS = 2000 
crop_count = 0

print("Extracting vehicle crops and computing VGG features...")

with torch.no_grad():
    for img_info in tqdm(ann["images"]):
        if crop_count >= MAX_CROPS:
            break
            
        img_id = img_info["id"]
        
        # Skip if this image has no vehicles annotated
        if img_id not in img_id_to_anns:
            continue
            
        img_path = os.path.join(dataset_dir, "train", img_info["file_name"])
        
        try:
            img = Image.open(img_path).convert("RGB")
        except Exception as e:
            print(f"Error loading {img_path}: {e}")
            continue
            
        # Process every vehicle bounding box in the current image
        for a in img_id_to_anns[img_id]:
            bbox = a["bbox"]  # COCO format: [x_min, y_min, width, height]
            x_min, y_min, w, h = bbox
            
            # Filter out impossibly small or corrupted bounding boxes (e.g., less than 10x10 pixels)
            # This prevents errors when resizing to 224x224
            if w < 10 or h < 10: 
                continue
                
            # Convert to PIL format: (left, top, right, bottom)
            x_max = x_min + w
            y_max = y_min + h
            
            # 2. Crop the vehicle!
            crop_img = img.crop((x_min, y_min, x_max, y_max))
            
            # 3. Apply the VGG transforms (Resize to 224x224, convert to tensor, normalize)
            crop_tensor = transform(crop_img).unsqueeze(0).to(device)
            
            # 4. Pass through VGG to get features
            feat = vgg_part(crop_tensor).cpu().numpy()
            
            all_crop_features.append(feat)
            crop_labels.append(a["category_id"])
            crop_count += 1
            
            if crop_count >= MAX_CROPS:
                break

# 5. Compute the new Delta Hyperbolicity
if len(all_crop_features) > 0:
    all_crop_features = np.concatenate(all_crop_features, axis=0)
    print(f"\nTotal valid vehicle crops extracted: {len(all_crop_features)}")
    
    mean_delta, std_delta = batched_delta_hyp(
        all_crop_features, 
        n_tries=20, 
        batch_size=1000
    )
    print(f"\n========================================================")
    print(f"Relative delta hyperbolicity (Cropped VGG): {mean_delta:.6f} +/- {std_delta:.6f}")
    print(f"========================================================")
else:
    print("No crops were extracted. Check your dataset paths.")

Extracting vehicle crops and computing VGG features...


  7%|▋         | 381/5528 [04:00<54:11,  1.58it/s]  



Total valid vehicle crops extracted: 2000


100%|██████████| 20/20 [15:09<00:00, 45.47s/it]


Relative delta hyperbolicity (Cropped VGG): 0.253886 +/- 0.019167


### 4. Class-Level Relative Delta

In [13]:
import numpy as np
from scipy.spatial import distance_matrix

class_names = ['cars-counter', 'Bus', 'Motorcycle', 'Pickup', 'Sedan', 'Suv', 'Truck', 'Van']
n_classes = len(class_names)

# 1. Convert the crop_labels list from the previous step into a numpy array
crop_labels_np = np.array(crop_labels)

# 2. Compute average feature centroid per class
centroids = []
for cls_id in range(n_classes):
    # Use the new crop_labels_np instead of the old all_labels
    mask = crop_labels_np == cls_id
    
    if mask.sum() > 0:
        # Calculate the mean of the features for this specific class
        centroids.append(all_crop_features[mask].mean(axis=0))
    else:
        # If a class wasn't found in the 2000 crops, append zeros to avoid errors
        centroids.append(np.zeros(all_crop_features.shape[1]))

centroids = np.array(centroids)

# 3. Calculate distance matrix and delta
class_distmat = distance_matrix(centroids, centroids)

delta = delta_hyp(class_distmat)
diam = np.max(class_distmat)
relative_delta = 2 * delta / diam

print(f"Class-level relative delta (VGG Crops): {relative_delta:.6f}")
print(f"  (delta={delta:.4f}, diameter={diam:.4f})")

Class-level relative delta (VGG Crops): 0.215123
  (delta=2.9772, diameter=27.6791)
